# IIR Low-Pass Filter on IQ Signal: Pseudocode → Code

**Prompt source:** `prompts/iir-pseudocode-to-code.md`  
**Skills used:** `pseudocode`, `python-notebook`

This notebook follows a two-stage approach:
1. Express the algorithm in language-agnostic pseudocode (pseudocode skill conventions)
2. Translate each pseudocode block 1-to-1 into Python / NumPy / SciPy

---
## Stage 1 — Pseudocode

Conventions (pseudocode skill):
- `←` for assignment, `=` for comparison
- PascalCase for algorithms/functions, camelCase for variables
- All blocks closed explicitly (`END FUNCTION`, `END FOR`, `END IF`)
- Arrays are zero-indexed

```
─────────────────────────────────────────────────────────────
FUNCTION GenerateSyntheticIQ(N, L, seed)
    DECLARE rng   AS RandomGenerator
    DECLARE X     AS ARRAY of FLOAT32, shape (N, 2, L)
    rng ← RandomGenerator(seed)
    X   ← rng.standardNormal(shape=(N, 2, L)) cast to FLOAT32
    RETURN X
END FUNCTION

─────────────────────────────────────────────────────────────
FUNCTION DesignButterworthIIR(cutoff, sampleRate, order)
    // cutoff is normalised: 0 < cutoff < 1  (1 = Nyquist)
    DECLARE b AS ARRAY   // numerator coefficients
    DECLARE a AS ARRAY   // denominator coefficients
    (b, a) ← ButterworthLowPass(order, cutoff, btype='low')
    RETURN (b, a)
END FUNCTION

─────────────────────────────────────────────────────────────
FUNCTION ApplyIIRFilter(X, b, a)
    // Applies the IIR filter along axis 2 (time) for every example
    DECLARE Xf AS ARRAY, shape same as X
    Xf ← lfilter(b, a, X, axis=2)
    RETURN Xf
END FUNCTION

─────────────────────────────────────────────────────────────
FUNCTION ComputePower(X)
    // X shape: (N, 2, L)  — axis 1: index 0=I, index 1=Q
    DECLARE I AS ARRAY ← X[:, 0, :]
    DECLARE Q AS ARRAY ← X[:, 1, :]
    DECLARE power AS FLOAT
    power ← mean(I² + Q²)
    RETURN power
END FUNCTION

─────────────────────────────────────────────────────────────
--- Main ---

// Parameters
N          ← 5
L          ← 1000
SEED       ← 42
CUTOFF     ← 0.1     // normalised frequency (fraction of Nyquist)
ORDER      ← 2
SAMPLERATE ← 1.0

// Generate signal
X ← GenerateSyntheticIQ(N, L, SEED)
ASSERT X.shape = (N, 2, L)

// Design filter
(b, a) ← DesignButterworthIIR(CUTOFF, SAMPLERATE, ORDER)

// Apply filter
Xf ← ApplyIIRFilter(X, b, a)
ASSERT Xf.shape = X.shape

// Compare power
pBefore ← ComputePower(X)
pAfter  ← ComputePower(Xf)
ASSERT pAfter < pBefore   // low-pass filter reduces total power

PRINT 'Power before filtering:', pBefore
PRINT 'Power after  filtering:', pAfter
PRINT 'Power reduction (dB):  ', 10 * log10(pAfter / pBefore)
─────────────────────────────────────────────────────────────
```

---
## Stage 2 — Python / NumPy / SciPy Translation

Each function below maps 1-to-1 to its pseudocode counterpart.

### Imports

In [ ]:
import numpy as np
from scipy.signal import butter, lfilter, freqz
import matplotlib.pyplot as plt

### `GenerateSyntheticIQ` → Python

In [ ]:
def generate_synthetic_iq(N: int, L: int, seed: int) -> np.ndarray:
    """FUNCTION GenerateSyntheticIQ(N, L, seed)
    Returns X of shape (N, 2, L), dtype float32.
    Axis 0 = examples, axis 1 = [I, Q], axis 2 = time samples.
    """
    rng = np.random.default_rng(seed)                          # rng ← RandomGenerator(seed)
    X   = rng.standard_normal((N, 2, L)).astype(np.float32)   # X   ← rng.standardNormal(...)
    return X                                                   # RETURN X

### `DesignButterworthIIR` → Python

In [ ]:
def design_butterworth_iir(cutoff: float, sample_rate: float, order: int):
    """FUNCTION DesignButterworthIIR(cutoff, sampleRate, order)
    cutoff is normalised: 0 < cutoff < 1  (1 = Nyquist).
    Returns (b, a) coefficient arrays.
    """
    nyquist  = sample_rate / 2.0
    wn       = cutoff / nyquist          # normalised cutoff for butter()
    b, a     = butter(order, wn, btype='low')   # (b, a) ← ButterworthLowPass(...)
    return b, a                          # RETURN (b, a)

### `ApplyIIRFilter` → Python

In [ ]:
def apply_iir_filter(X: np.ndarray, b: np.ndarray, a: np.ndarray) -> np.ndarray:
    """FUNCTION ApplyIIRFilter(X, b, a)
    Applies IIR filter along axis=2 (time samples) for every example.
    Returns Xf of same shape as X.
    """
    Xf = lfilter(b, a, X, axis=2)   # Xf ← lfilter(b, a, X, axis=2)
    return Xf                        # RETURN Xf

### `ComputePower` → Python

In [ ]:
def compute_power(X: np.ndarray) -> float:
    """FUNCTION ComputePower(X)
    X shape: (N, 2, L). Returns mean power across all samples.
    """
    I     = X[:, 0, :]              # I ← X[:, 0, :]
    Q     = X[:, 1, :]              # Q ← X[:, 1, :]
    power = np.mean(I**2 + Q**2)   # power ← mean(I² + Q²)
    return float(power)             # RETURN power

### Main — execute the pipeline

In [ ]:
# ── Parameters ──────────────────────────────────────────────
N           = 5
L           = 1000
SEED        = 42
CUTOFF      = 0.1    # normalised frequency (fraction of Nyquist)
ORDER       = 2
SAMPLE_RATE = 1.0

# ── Generate signal ─────────────────────────────────────────
X = generate_synthetic_iq(N, L, SEED)
assert X.shape == (N, 2, L), f'Expected ({N}, 2, {L}), got {X.shape}'
print(f'X shape  : {X.shape}  (N, 2, L)')

# ── Design filter ────────────────────────────────────────────
b, a = design_butterworth_iir(CUTOFF, SAMPLE_RATE, ORDER)
print(f'b coeffs : {np.round(b, 6)}')
print(f'a coeffs : {np.round(a, 6)}')

# ── Apply filter ─────────────────────────────────────────────
Xf = apply_iir_filter(X, b, a)
assert Xf.shape == X.shape, f'Shape mismatch: {Xf.shape} vs {X.shape}'
print(f'Xf shape : {Xf.shape}  — shape preserved ✓')

# ── Compare power ────────────────────────────────────────────
p_before = compute_power(X)
p_after  = compute_power(Xf)
assert p_after < p_before, 'Expected power reduction after low-pass filtering'

print()
print(f'Power before filtering : {p_before:.6f}')
print(f'Power after  filtering : {p_after:.6f}')
print(f'Power reduction (dB)   : {10 * np.log10(p_after / p_before):.3f} dB')

---
## Validation — Filter Frequency Response

In [ ]:
w, h = freqz(b, a, worN=512)
freqs = w / np.pi  # normalised 0→1 (Nyquist)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(freqs, 20 * np.log10(np.abs(h) + 1e-12), color='steelblue', linewidth=1.5)
ax.axvline(x=CUTOFF, color='tomato', linestyle='--', label=f'Cutoff = {CUTOFF} (normalised)')
ax.axhline(y=-3, color='gray', linestyle=':', label='-3 dB')
ax.set_xlabel('Normalised frequency (1 = Nyquist)')
ax.set_ylabel('Magnitude (dB)')
ax.set_title(f'Butterworth IIR Low-Pass — order={ORDER}, cutoff={CUTOFF}')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(-80, 5)
plt.tight_layout()
plt.show()

## Validation — I Channel: Before vs After (Example 0)

In [ ]:
t = np.arange(L)

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

axes[0].plot(t, X[0, 0, :], color='steelblue', linewidth=0.7, label='I channel — raw')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Raw IQ Signal — Example 0, I channel')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(t, Xf[0, 0, :], color='tomato', linewidth=0.7,
             label=f'I channel — IIR filtered (Butterworth order={ORDER}, cutoff={CUTOFF})')
axes[1].set_xlabel('Sample index')
axes[1].set_ylabel('Amplitude')
axes[1].set_title('Filtered IQ Signal — Example 0, I channel')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Summary

| Pseudocode Function | Python Translation | Key construct |
|---|---|---|
| `GenerateSyntheticIQ` | `generate_synthetic_iq()` | `np.random.default_rng` |
| `DesignButterworthIIR` | `design_butterworth_iir()` | `scipy.signal.butter` |
| `ApplyIIRFilter` | `apply_iir_filter()` | `scipy.signal.lfilter(axis=2)` |
| `ComputePower` | `compute_power()` | `np.mean(I² + Q²)` |

**Assertions passed:**
- `X.shape == (N, 2, L)` ✓
- `Xf.shape == X.shape` ✓
- `p_after < p_before` ✓ — low-pass filter attenuates high-frequency energy

**Next steps:**
- Sweep `ORDER` (2, 4, 6) and compare roll-off steepness
- Sweep `CUTOFF` (0.05 → 0.5) and observe power reduction
- Replace `lfilter` with `sosfilt` for better numerical stability at higher orders